# 04. Convergent Validity With Independent Observer Ratings (Appendix E.4; Exp 14–15)

In [1]:
import os, sys, json
import numpy as np, pandas as pd
sys.path.insert(0, os.path.abspath("."))
import common as C
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 40)
rng = np.random.default_rng(C.SEED)
print("final data folder:", os.path.relpath(C.FINAL))

final data folder: ../final


## 1. Load the cleaned observer tables

In [2]:
LONG = pd.read_csv(os.path.join(C.REPRO, "observer_ratings_long_clean.csv"))
OC = C.load_observer_composites()
ALL = pd.read_csv(os.path.join(C.OUT, "labels_all.csv")); truth = ALL[ALL.exp.isin(C.MLLM_EXPS + C.ML_EXPS)].drop_duplicates("pid")[["pid", "true"]]
RATERS = ["Rater A", "Rater B", "Rater C"]
print("rating rows:", len(LONG), "| raters:", LONG.rater.unique().tolist(), "| versions:", LONG.version.unique().tolist())
OC.head()

rating rows: 611 | raters: ['Rater A', 'Rater B', 'Rater C'] | versions: ['audiovisual', 'silent']
Out[2]: 
   pid version construct  Rater A  Rater B  Rater C  consensus_complete_case  consensus_available_raters
0    1  silent       SYM     1.50     2.00      1.5                 1.666667                    1.666667
1    2  silent       SYM     1.75     1.50      2.0                 1.750000                    1.750000
2    3  silent       SYM     1.00     1.75      2.0                 1.583333                    1.583333
3    4  silent       SYM     1.25     1.00      2.0                 1.416667                    1.416667
4    5  silent       SYM     1.25     1.00      2.0                 1.416667                    1.416667


## 2. E.4.2 Inter-rater reliability

In [3]:
def panel(version, construct):
    w = OC[(OC.version == version) & (OC.construct == construct)].set_index("pid")[RATERS].dropna(); return w
rows = []; boots = {}
for ver in ("silent", "audiovisual"):
    for con, name in [("SYM", "Sympathy"), ("INS", "Inspiration")]:
        X = panel(ver, con).values; r = C.icc_mw(X); b = C.icc_boot(X, np.random.default_rng(C.SEED)); boots[(ver, con)] = b
        for k, lab in [("A1", "ICC(A,1)"), ("Ak", "ICC(A,k)"), ("C1", "ICC(C,1)")]:
            rows.append(dict(section="icc", statistic=f"{lab} {ver} {name}", value=r[k], ci_lo=np.percentile(b[k], 2.5), ci_hi=np.percentile(b[k], 97.5), n=len(X)))
# audiovisual minus silent, on the participants present in both panels
for con, name in [("SYM", "Sympathy"), ("INS", "Inspiration")]:
    a = panel("audiovisual", con); s = panel("silent", con); common = a.index.intersection(s.index)
    rng_ = np.random.default_rng(C.SEED); n = len(common); idx = rng_.integers(0, n, (C.NB, n)); A_ = a.loc[common].values; S_ = s.loc[common].values
    for k, lab in [("Ak", "ICC(A,k)"), ("A1", "ICC(A,1)")]:
        diffs = np.array([C.icc_mw(A_[i])[k] - C.icc_mw(S_[i])[k] for i in idx])
        rows.append(dict(section="icc_diff", statistic=f"{lab} audiovisual minus silent {name}", value=C.icc_mw(A_)[k] - C.icc_mw(S_)[k], ci_lo=np.percentile(diffs, 2.5), ci_hi=np.percentile(diffs, 97.5), n=n))
ICC = pd.DataFrame(rows); ICC.round(3)

Out[3]: 
     section                                      statistic  value  ci_lo  ci_hi    n
0        icc                       ICC(A,1) silent Sympathy  0.185  0.100  0.269  101
1        icc                       ICC(A,k) silent Sympathy  0.405  0.250  0.525  101
2        icc                       ICC(C,1) silent Sympathy  0.320  0.192  0.433  101
3        icc                    ICC(A,1) silent Inspiration  0.390  0.260  0.507  100
4        icc                    ICC(A,k) silent Inspiration  0.657  0.513  0.755  100
5        icc                    ICC(C,1) silent Inspiration  0.414  0.279  0.533  100
6        icc                  ICC(A,1) audiovisual Sympathy  0.578  0.472  0.668  100
7        icc                  ICC(A,k) audiovisual Sympathy  0.804  0.728  0.858  100
8        icc                  ICC(C,1) audiovisual Sympathy  0.638  0.552  0.710  100
9        icc               ICC(A,1) audiovisual Inspiration  0.535  0.430  0.630  101
10       icc               ICC(A,k) audiovisu

Per-item ICC(A,1), on the rating panel (participants scored by all three raters in both versions). In the silent version the vocal and linguistic items carry almost no variance, so their coefficients are near zero or not estimable.

In [4]:
ITEMS = ["SYM_F", "SYM_V", "SYM_L", "SYM_B", "INS_F", "INS_V", "INS_L", "INS_B"]; irows = []
with np.errstate(invalid="ignore", divide="ignore"):
    for ver in ("silent", "audiovisual"):
        for it in ITEMS:
            X = LONG[(LONG.version == ver) & LONG.in_rating_panel].pivot_table(index="pid", columns="rater", values=it, aggfunc="first")[RATERS].dropna()
            irows.append(dict(version=ver, item=it.replace("_", "-"), n=len(X), icc_a1=C.icc_mw(X.values)["A1"]))
ITEM = pd.DataFrame(irows); ITEM.to_csv(os.path.join(C.OUT, "observer_item_icc.csv"), index=False)
ITEM.pivot(index="item", columns="version", values="icc_a1").round(3)

Out[4]: 
version  audiovisual  silent
item                        
INS-B          0.350   0.177
INS-F          0.422   0.392
INS-L          0.669     NaN
INS-V          0.207     NaN
SYM-B          0.127   0.078
SYM-F          0.355   0.170
SYM-L          0.721  -0.000
SYM-V          0.414  -0.000


## 3. E.4.3 Manipulation check on the observer scores

In [5]:
from scipy import stats
cons = OC[OC.version == "audiovisual"].pivot(index="pid", columns="construct", values="consensus_complete_case").dropna().reset_index().merge(truth, on="pid")
cons["S_hi"] = cons.true.map(C.SYM_HI); cons["I_hi"] = cons.true.map(C.INS_HI)
MANIP = {}
for con, fac in [("SYM", "S_hi"), ("INS", "I_hi")]:
    hi = cons.loc[cons[fac] == 1, con].values; lo = cons.loc[cons[fac] == 0, con].values
    U, p = stats.mannwhitneyu(hi, lo, alternative="two-sided"); d, g = C.cohen_d(hi, lo)
    MANIP[con] = dict(mean_hi=hi.mean(), mean_lo=lo.mean(), n_hi=len(hi), n_lo=len(lo), U=U, p=p, d=d, g=g)
    rows.append(dict(section="manip_primary_n100", statistic=f"{con} audiovisual consensus, high vs low {fac}: Mann-Whitney U", value=U, n=len(cons)))
    rows.append(dict(section="manip_primary_n100", statistic=f"{con} audiovisual consensus, high vs low {fac}: Cohen d", value=d, n=len(cons)))
    rows.append(dict(section="manip_primary_n100", statistic=f"{con} audiovisual consensus, high vs low {fac}: Hedges g", value=g, n=len(cons)))
    print(f"{con}: mean {hi.mean():.2f} (n = {len(hi)}) vs {lo.mean():.2f} (n = {len(lo)}), U = {U:.1f}, p {C.fmt_p(p)}, d = {d:.2f}, g = {g:.2f}, n = {len(cons)}")
CELL = cons.groupby("true")[["SYM", "INS"]].mean().loc[C.CLS]
print("\nconsensus means by induction cell:"); print(CELL.round(2))
AN_OBS = C.anova_2x2(cons.rename(columns={"SYM": "symp", "INS": "insp"}), 15); AN_OBS.to_csv(os.path.join(C.OUT, "observer_anova.csv"), index=False)
print("\nType II 2 x 2 ANOVA of the audiovisual consensus scores (n = %d):" % len(cons)); print(AN_OBS[["score", "effect", "df1", "df2", "F", "p", "eta_p2"]].round(3).to_string(index=False))

SYM: mean 2.44 (n = 49) vs 1.18 (n = 51), U = 2250.5, p < .001, d = 1.61, g = 1.60, n = 100
INS: mean 2.20 (n = 47) vs 1.13 (n = 53), U = 2303.0, p < .001, d = 1.81, g = 1.80, n = 100

consensus means by induction cell:
       SYM   INS
true            
ELVT  2.14  1.90
SYMP  2.73  1.21
INSP  1.39  2.52
CTRL  1.01  1.07

Type II 2 x 2 ANOVA of the audiovisual consensus scores (n = 100):
score            effect  df1  df2      F     p  eta_p2
 symp    Sympathy manip    1   96 70.819 0.000   0.425
 symp Inspiration manip    1   96  0.401 0.528   0.004
 symp       Interaction    1   96 10.591 0.002   0.099
 insp    Sympathy manip    1   96  3.942 0.050   0.039
 insp Inspiration manip    1   96 94.570 0.000   0.496
 insp       Interaction    1   96 11.955 0.001   0.111


## 4. E.4.4 Agreement between observers and the model

In [6]:
cons_s = OC[OC.version == "silent"].pivot(index="pid", columns="construct", values="consensus_complete_case").dropna().reset_index()
SP = []
for ver, ref in [("audiovisual", cons[["pid", "SYM", "INS"]]), ("silent", cons_s[["pid", "SYM", "INS"]])]:
    for e in C.MLLM_EXPS:
        d = ALL[ALL.exp == e].merge(ref, on="pid")
        for con, ms, name in [("SYM", "symp", "Sympathy"), ("INS", "insp", "Inspiration")]:
            rho, p, lo, hi = C.spearman_with_ci(d[ms].values, d[con].values)
            SP.append(dict(version=ver, construct=name, exp=e, configuration=C.NAMES[e], n=len(d), rho=rho, ci_lo=lo, ci_hi=hi, p=p))
            if ver == "audiovisual": rows.append(dict(section="spearman_audiovisual", statistic=f"rho Exp {e} {name}", value=rho, ci_lo=lo, ci_hi=hi, n=len(d)))
SP = pd.DataFrame(SP); SP.to_csv(os.path.join(C.OUT, "observer_model_spearman.csv"), index=False)
sil = SP[SP.version == "silent"]
print(f"silent consensus: rho from {sil.rho.min():.3f} to {sil.rho.max():.3f}; {int((sil.p < .05).sum())} of {len(sil)} with p < .05")
SP[SP.version == "audiovisual"].pivot(index="exp", columns="construct", values=["n", "rho", "ci_lo", "ci_hi"]).round(2)

silent consensus: rho from -0.048 to 0.380; 12 of 20 with p < .05
Out[6]: 
                    n                  rho                ci_lo                ci_hi         
construct Inspiration Sympathy Inspiration Sympathy Inspiration Sympathy Inspiration Sympathy
exp                                                                                          
1                94.0     94.0        0.28     0.17        0.15    -0.05        0.42     0.38
2                98.0     98.0        0.49     0.69        0.31     0.56        0.65     0.80
3                98.0     98.0        0.71     0.74        0.58     0.61        0.81     0.84
4                98.0     98.0        0.72     0.76        0.60     0.66        0.82     0.83
5                98.0     98.0        0.75     0.80        0.63     0.71        0.84     0.87
6                98.0     98.0        0.80     0.84        0.71     0.75        0.86     0.90
7                98.0     98.0        0.67     0.61        0.54     0.44       

## 5. E.4.5 Agreement of individual and pooled human raters with the model

The reference for both a human rater and the model is the consensus (mean) of the *other two* raters, so the two are compared against an identical standard; the model is the agentic configuration (Exp 9). The sample is the participants with all three audiovisual ratings and an Exp 9 score. The intervals for the difference (mean model correlation minus mean individual-rater correlation) and for the model-consensus correlation come from 2,000 participant resamples drawn from one generator (seed 42), Sympathy first. The scale-use descriptives of the last paragraph of E.4.5 (model Sympathy mean and SD; each rater's mean and SD) are computed on the same 97 participants as the Sympathy comparison.

In [7]:
d9 = ALL[ALL.exp == 9][["pid", "symp", "insp"]]
def rater_vs_model(H, m):
    hr, mr = [], []
    for k in range(H.shape[1]):
        others = np.delete(H, k, axis=1).mean(1)
        hr.append(stats.spearmanr(H[:, k], others)[0]); mr.append(stats.spearmanr(m, others)[0])
    return np.array(hr), np.array(mr), stats.spearmanr(m, H.mean(1))[0]
HVM = []; rb = np.random.default_rng(C.SEED)          # one generator for both constructs (Sympathy resampled first)
for con, ms, name in [("SYM", "symp", "Sympathy"), ("INS", "insp", "Inspiration")]:
    w = OC[(OC.version == "audiovisual") & (OC.construct == con)].set_index("pid")[RATERS].dropna().reset_index().merge(d9, on="pid")
    H = w[RATERS].values; m = w[ms].values; n = len(w)
    hr, mr, full = rater_vs_model(H, m)
    diffs, fulls = [], []
    for _ in range(2000):
        i = rb.integers(0, n, n); a, b, f = rater_vs_model(H[i], m[i]); diffs.append(b.mean() - a.mean()); fulls.append(f)
    HVM.append(dict(construct=name, n=n, rater_min=hr.min(), rater_max=hr.max(), rater_mean=hr.mean(), model_min=mr.min(), model_max=mr.max(), model_mean=mr.mean(),
                    diff=mr.mean() - hr.mean(), diff_lo=np.nanpercentile(diffs, 2.5), diff_hi=np.nanpercentile(diffs, 97.5),
                    model_vs_consensus=full, mvc_lo=np.nanpercentile(fulls, 2.5), mvc_hi=np.nanpercentile(fulls, 97.5)))
HVM = pd.DataFrame(HVM); HVM.to_csv(os.path.join(C.OUT, "observer_human_vs_model.csv"), index=False); print(HVM.round(3).T.to_string())
# scale use (E.4.5, last paragraph): the 97 participants of the Sympathy comparison (all three audiovisual Sympathy ratings and an Exp 9 score)
wa = OC[(OC.version == "audiovisual") & (OC.construct == "SYM")].set_index("pid")[RATERS].dropna().reset_index().merge(d9, on="pid")
SCALE = dict(model_mean=wa.symp.mean(), model_sd=wa.symp.std(ddof=1), model_n=len(wa),
             rater_means=[wa[r].mean() for r in RATERS], rater_sds=[wa[r].std(ddof=1) for r in RATERS])
print(f"Sympathy scale use (n = {len(wa)}): model {SCALE['model_mean']:.2f} (SD {SCALE['model_sd']:.2f}); raters", ", ".join(f"{a:.2f} (SD {b:.2f})" for a, b in zip(SCALE["rater_means"], SCALE["rater_sds"])))

                           0            1
construct           Sympathy  Inspiration
n                         97           98
rater_min              0.771        0.741
rater_max              0.791        0.782
rater_mean             0.783        0.764
model_min              0.804        0.726
model_max              0.841        0.765
model_mean             0.828        0.748
diff                   0.045       -0.015
diff_lo               -0.023       -0.139
diff_hi                0.117        0.087
model_vs_consensus     0.854        0.782
mvc_lo                  0.78         0.67
mvc_hi                 0.907        0.869
Sympathy scale use (n = 97): model 4.25 (SD 3.43); raters 1.55 (SD 1.05), 1.65 (SD 0.93), 2.24 (SD 1.45)


## 6. E.4.6 Human observer benchmark: consensus and individual raters

In [8]:
for ver, e in [("silent", 14), ("audiovisual", 15)]:
    panel_pids = OC[OC.version == ver].pivot(index="pid", columns="construct", values="consensus_complete_case").dropna().index
    out = {}
    for rt in ["consensus_complete_case"] + RATERS:
        w = OC[OC.version == ver].pivot(index="pid", columns="construct", values=rt).loc[panel_pids].dropna().reset_index().merge(truth, on="pid")
        pred = C.mean_threshold_labels(w.SYM, w.INS); acc = float((pred == w.true.values).mean()); out[rt] = (round(acc, 3), len(w))
        rows.append(dict(section="classification_primary", statistic=f"Exp {e} {ver} {'' if rt.startswith('consensus') else rt + ' '}four-class accuracy", value=acc, n=len(w)))
    print(f"Exp {e} ({ver}) four-class accuracy on the {len(panel_pids)}-participant consensus panel (value, n):", out)
OBS = pd.DataFrame(rows); OBS.to_csv(os.path.join(C.OUT, "observer_statistics.csv"), index=False)

Exp 14 (silent) four-class accuracy on the 100-participant consensus panel (value, n): {'consensus_complete_case': (0.31, 100), 'Rater A': (0.26, 100), 'Rater B': (0.29, 100), 'Rater C': (0.26, 100)}
Exp 15 (audiovisual) four-class accuracy on the 100-participant consensus panel (value, n): {'consensus_complete_case': (0.73, 100), 'Rater A': (0.66, 100), 'Rater B': (0.66, 100), 'Rater C': (0.72, 100)}


## 7. Verification

ICC, manipulation-check, ANOVA, Spearman, per-item ICC and benchmark statistics are compared with the frozen file `observer_stats_frozen.csv` (its statistic labels differ from the ones used here, so the pairs are matched explicitly). The E.4.5 comparison and the scale-use descriptives have no frozen counterpart and are compared with the values printed in Appendix E.4.5 (two decimals).

In [9]:
fz = C.load_frozen("observer_stats_frozen.csv")
def key(s): return " ".join(str(s).replace("Inspiration", "INS").replace("Sympathy", "SYM").replace(",", "").split()[:6])
a = OBS[OBS.section.isin(["icc", "icc_diff", "classification_primary"])]; a = a.assign(k=a.section + "|" + a.statistic.map(key)); b = fz.assign(k=fz.section + "|" + fz.statistic.map(key))
m = a.merge(b[["k", "value", "ci_lo", "ci_hi"]], on="k", suffixes=("", "_frozen"))
print("ICC / ICC difference / benchmark: matched", len(m), "of", len(a), "| max |diff| point:", float((m.value - m.value_frozen).abs().max().round(6)),
      "| max |diff| CI bound:", float(np.nanmax(np.r_[(m.ci_lo - m.ci_lo_frozen).abs(), (m.ci_hi - m.ci_hi_frozen).abs()]).round(4)))
pairs = []
for con, name in [("SYM", "Sympathy"), ("INS", "Inspiration")]:
    hi_lab = "high-symp" if con == "SYM" else "high-insp"; lo_lab = hi_lab.replace("high", "low")
    pairs += [(MANIP[con]["mean_hi"], f"observed {name} mean, {hi_lab} cells"), (MANIP[con]["mean_lo"], f"observed {name} mean, {lo_lab} cells"),
              (MANIP[con]["d"], f"observed {name} Cohen d"), (MANIP[con]["g"], f"observed {name} Hedges g"),
              (MANIP[con]["U"], f"observed {name} Mann-Whitney U"), (MANIP[con]["p"], f"observed {name} Mann-Whitney p")]
    pairs += [(CELL.loc[c, con], f"cell mean observed {name}, {c}") for c in C.CLS]
    sc = "symp" if con == "SYM" else "insp"
    for eff, lab in [("Sympathy manip", "C(symp_hi)"), ("Inspiration manip", "C(insp_hi)"), ("Interaction", "C(symp_hi):C(insp_hi)")]:
        pairs.append((AN_OBS[(AN_OBS.score == sc) & (AN_OBS.effect == eff)].F.iloc[0], f"ANOVA type II, {name} ~ {lab}: F"))
FZD = {(r.section, r.statistic): r.value for r in fz.itertuples()}
dm = max(abs(float(v) - float(FZD[("manip_primary_n100", s)])) for v, s in pairs)
print("E.4.3 manipulation check and ANOVA F: compared", len(pairs), "| max |diff|:", float(np.round(dm, 6)))
sp_f = fz[fz.section.isin(["spearman_audiovisual", "spearman_silent"])].dropna(subset=["value"]).copy()
sp_f["version"] = sp_f.section.str.replace("spearman_", ""); sp_f["exp"] = sp_f.statistic.str.extract(r"Exp (\d+)").astype(float); sp_f["construct"] = np.where(sp_f.statistic.str.contains("Inspiration|INS"), "Inspiration", "Sympathy")
mm = SP.merge(sp_f[["version", "exp", "construct", "value", "ci_lo", "ci_hi", "n"]], on=["version", "exp", "construct"], suffixes=("", "_frozen"))
print("E.4.4 Spearman rho (audiovisual + silent): matched", len(mm), "of", len(SP), "| max |diff| rho", float((mm.rho - mm.value).abs().max().round(6)), "| n agreement", int((mm.n == mm.n_frozen).sum()))
mav = mm[mm.version == "audiovisual"]; print("   audiovisual CI bounds max |diff|:", float(max((mav.ci_lo - mav.ci_lo_frozen).abs().max(), (mav.ci_hi - mav.ci_hi_frozen).abs().max()).round(4)))
fi = fz[fz.section == "icc_item"].copy(); fi["version"] = fi.statistic.str.split().str[1]; fi["item"] = fi.statistic.str.split().str[2]
mi = ITEM.merge(fi[["version", "item", "value", "n"]], on=["version", "item"], suffixes=("", "_frozen")).dropna(subset=["value"])
print("E.4.2 per-item ICC(A,1): matched", len(mi), "| max |diff|:", float((mi.icc_a1 - mi.value).abs().max().round(6)), "| n agreement", int((mi.n == mi.n_frozen).sum()))
E45 = {"Sympathy": dict(n=97, rater_min=.77, rater_max=.79, rater_mean=.78, model_min=.80, model_max=.84, model_mean=.83, diff=.04, diff_lo=-.02, diff_hi=.12, model_vs_consensus=.85, mvc_lo=.78, mvc_hi=.91),
       "Inspiration": dict(n=98, rater_min=.74, rater_max=.78, rater_mean=.76, model_min=.73, model_max=.77, model_mean=.75, diff=-.02, diff_lo=-.14, diff_hi=.09, model_vs_consensus=.78, mvc_lo=.67, mvc_hi=.87)}
ms = pd.DataFrame(E45).T.rename_axis("construct").reset_index()
print(C.compare(HVM, ms, ["construct"], list(E45["Sympathy"]), atol=0.011).to_string(index=False))
sc_rec = [round(float(x), 2) for x in (SCALE["model_mean"], SCALE["model_sd"], min(SCALE["rater_means"]), max(SCALE["rater_means"]), min(SCALE["rater_sds"]), max(SCALE["rater_sds"]))]
sc_ms = [4.25, 3.43, 1.55, 2.24, 0.93, 1.45]
print("E.4.5 scale use, n =", SCALE["model_n"], "| recomputed:", sc_rec, "| manuscript (model mean, SD; rater mean min, max; rater SD min, max):", sc_ms, "| match:", bool(np.all(np.abs(np.array(sc_rec) - np.array(sc_ms)) <= 0.005 + 1e-9)))

ICC / ICC difference / benchmark: matched 24 of 24 | max |diff| point: 0.0 | max |diff| CI bound: 0.0
E.4.3 manipulation check and ANOVA F: compared 26 | max |diff|: 0.0
E.4.4 Spearman rho (audiovisual + silent): matched 40 of 40 | max |diff| rho 0.0 | n agreement 40
   audiovisual CI bounds max |diff|: 0.0
E.4.2 per-item ICC(A,1): matched 14 | max |diff|: 0.0 | n agreement 14
            column  n_compared  max_abs_diff  within_tolerance
                 n           2      0.000000              True
         rater_min           2      0.001345              True
         rater_max           2      0.002111              True
        rater_mean           2      0.003537              True
         model_min           2      0.004452              True
         model_max           2      0.005164              True
        model_mean           2      0.002323              True
              diff           2      0.004822              True
           diff_lo           2      0.002795         